# 15 — Importance Weighting via Propensity Score (correção formal de covariate shift)
## PRT Seguros

Técnica clássica de domain adaptation (Shimodaira 2000 / Sugiyama): se treino e teste vêm de
distribuições diferentes P_treino(X) ≠ P_teste(X), mas a relação P(y|X) é a mesma, dá pra corrigir
o viés dando **mais peso**, durante o treino, às linhas de treino que mais se parecem com o teste.

**Como calculamos o peso:** usamos o classificador adversarial do notebook `07` (que prevê "essa
linha é treino ou Kaggle?"). Para cada linha de treino, `p = P(ser Kaggle | features)`.
O peso de importância é `w = p / (1 - p)` — quanto mais uma linha de treino "parece" Kaggle,
mais peso ela ganha no treino do modelo de churn.

## 1. Carregar dados (usamos o conjunto v2: features selecionadas + engenharia)

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_predict
from sklearn.metrics import roc_auc_score
import lightgbm as lgb
from catboost import CatBoostClassifier

RANDOM_STATE = 42
ID_COL, TARGET_COL = "cod_individuo", "churned"

train = pd.read_csv("dados_processados/train_model_ready_v2.csv")
val = pd.read_csv("dados_processados/val_model_ready_v2.csv")
kaggle = pd.read_csv("dados_processados/kaggle_model_ready_v2.csv")

feature_cols = [c for c in train.columns if c not in (ID_COL, TARGET_COL)]
print(f"Treino: {train.shape} | Val: {val.shape} | Kaggle: {kaggle.shape}")


Treino: (80000, 81) | Val: (20000, 81) | Kaggle: (100000, 80)


## 2. Treinar o classificador adversarial (igual ao notebook `07`, mas só com o TREINO)

Importante: aqui usamos só `train` (não `val`) como "classe treino" — a validação continua de fora
de tudo, pra manter a métrica final honesta.

In [2]:
X_adv = pd.concat([train[feature_cols], kaggle[feature_cols]], ignore_index=True)
y_adv = np.concatenate([np.zeros(len(train)), np.ones(len(kaggle))])

adv_model = lgb.LGBMClassifier(
    n_estimators=300, max_depth=6, num_leaves=31, learning_rate=0.05,
    random_state=RANDOM_STATE, n_jobs=-1, verbose=-1,
)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
proba_oof = cross_val_predict(adv_model, X_adv, y_adv, cv=cv, method="predict_proba", n_jobs=1)[:, 1]
print(f"AUC adversarial (treino vs kaggle, OOF): {roc_auc_score(y_adv, proba_oof):.4f}")

# propensidade de cada linha de TREINO parecer Kaggle
propensao_treino = proba_oof[:len(train)]
propensao_treino = np.clip(propensao_treino, 0.01, 0.99)  # evita pesos infinitos

pesos_treino = propensao_treino / (1 - propensao_treino)
pesos_treino = pesos_treino / pesos_treino.mean()  # normaliza em torno de 1.0

print(f"Peso médio: {pesos_treino.mean():.2f} | mín: {pesos_treino.min():.2f} | máx: {pesos_treino.max():.2f}")
print(f"Percentis do peso: {np.percentile(pesos_treino, [5, 25, 50, 75, 95]).round(2)}")


AUC adversarial (treino vs kaggle, OOF): 0.6266
Peso médio: 1.00 | mín: 0.02 | máx: 46.20
Percentis do peso: [0.25 0.94 1.05 1.12 1.25]


## 3. Treinar CatBoost com e sem os pesos de importância

Mesmos hiperparâmetros tunados do notebook `11`. A única diferença é passar `sample_weight` no
`.fit()` da versão ponderada.

In [3]:
X_train, y_train = train[feature_cols], train[TARGET_COL]
X_val, y_val = val[feature_cols], val[TARGET_COL]
X_kaggle = kaggle[feature_cols]

MELHORES_PARAMS_CAT = {"random_strength": 0.5, "learning_rate": 0.02, "l2_leaf_reg": 9, "depth": 6}

# split de early stopping, carregando os pesos junto (mesmos índices)
idx_tr, idx_es = train_test_split(
    np.arange(len(X_train)), test_size=0.15, stratify=y_train, random_state=RANDOM_STATE
)
X_tr_es, y_tr_es, w_tr_es = X_train.iloc[idx_tr], y_train.iloc[idx_tr], pesos_treino[idx_tr]
X_es, y_es = X_train.iloc[idx_es], y_train.iloc[idx_es]
neg_pos_ratio_es = (y_tr_es == 0).sum() / (y_tr_es == 1).sum()

# --- SEM pesos (baseline, igual ao notebook 11) ---
cat_sem_peso = CatBoostClassifier(
    iterations=3000, **MELHORES_PARAMS_CAT, scale_pos_weight=neg_pos_ratio_es,
    eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
)
cat_sem_peso.fit(X_tr_es, y_tr_es, eval_set=(X_es, y_es))
auc_sem_peso = roc_auc_score(y_val, cat_sem_peso.predict_proba(X_val)[:, 1])

# --- COM pesos de importância (propensity score) ---
cat_com_peso = CatBoostClassifier(
    iterations=3000, **MELHORES_PARAMS_CAT, scale_pos_weight=neg_pos_ratio_es,
    eval_metric="AUC", random_seed=RANDOM_STATE, early_stopping_rounds=50, verbose=False,
)
cat_com_peso.fit(X_tr_es, y_tr_es, sample_weight=w_tr_es, eval_set=(X_es, y_es))
auc_com_peso = roc_auc_score(y_val, cat_com_peso.predict_proba(X_val)[:, 1])

print(f"CatBoost SEM importance weighting: AUC-ROC val = {auc_sem_peso:.4f}")
print(f"CatBoost COM importance weighting: AUC-ROC val = {auc_com_peso:.4f}")
print(f"Delta: {auc_com_peso - auc_sem_peso:+.4f}")


CatBoost SEM importance weighting: AUC-ROC val = 0.8263
CatBoost COM importance weighting: AUC-ROC val = 0.8252
Delta: -0.0011


## 4. Gerar submissão com o modelo vencedor

In [4]:
import os
os.makedirs("submissions", exist_ok=True)

melhor_modelo = cat_com_peso if auc_com_peso >= auc_sem_peso else cat_sem_peso
nome_melhor = "com_peso" if auc_com_peso >= auc_sem_peso else "sem_peso"
auc_melhor = max(auc_com_peso, auc_sem_peso)

proba_kaggle = melhor_modelo.predict_proba(X_kaggle)[:, 1]
submissao = pd.DataFrame({"Id": kaggle[ID_COL], "probabilidade_churn": proba_kaggle})
submissao.to_csv("submissions/submission_importance_weighting.csv", index=False)

print(f"Melhor versão: {nome_melhor} (AUC-ROC val = {auc_melhor:.4f})")
print("Salvo: submissions/submission_importance_weighting.csv")


Melhor versão: sem_peso (AUC-ROC val = 0.8263)
Salvo: submissions/submission_importance_weighting.csv
